In [0]:
from pyspark.sql.functions import lit

schema = "agent_id integer, agent_name string, agent_email string, agent_phone string, branch_id integer, create_timestamp timestamp"
df = spark.read.schema(schema).parquet("abfss://landing@projpolicysytem.dfs.core.windows.net/AgentData/*.parquet")
display(df)




In [0]:
df_with_flag = df.withColumn("merge_flag", lit(False))
bronze_path = "abfss://bronzelayer@projpolicysytem.dfs.core.windows.net/Agent"
df_with_flag.write \
    .format("delta") \
    .option("path", bronze_path) \
    .mode("append") \
    .saveAsTable("policyprojcatalog.policyprojdb.Agent")

In [0]:
%sql
select * from policyprojcatalog.policyprojdb.Agent

In [0]:
dbutils.fs.mv(
    "abfss://processed@projpolicysytem.dfs.core.windows.net/AgentData/",
    "abfss://landing@projpolicysytem.dfs.core.windows.net/AgentData/",
    True
)

In [0]:
from datetime import datetime

current_time = datetime.now().strftime('%m-%d-%Y')

new_folder = f"abfss://processed@projpolicysytem.dfs.core.windows.net/AgentData/{current_time}"

dbutils.fs.mv(
    "abfss://landing@projpolicysytem.dfs.core.windows.net/AgentData/",
    new_folder,
    True
)